# Week 2 — Tools & Permissions

*Course project: a personal assistant for **Alex, 44**: senior backend developer, full-time job, two kids,
retraining as an **AI engineer** while job-hunting.*

Last week the assistant could only **read files and do arithmetic**. This week it gets real tools over a calendar
and task database, and it gets the power to **change things**. That forces the questions this week is about:

1. **Granularity:** one powerful tool or many specific ones?
2. **Schemas:** how do we stop bad arguments before they do damage?
3. **Outputs:** what should a tool return, and how much?
4. **Errors:** which failures belong to the model, which to our code, which end the run?
5. **Permissions:** what may the agent do alone, what needs a human, what is forbidden?
6. **Parallelism:** which tool calls can run at the same time?
7. **Measurement:** does the model actually pick the right tool?

> Tools are the agent's user interface. The model only knows a tool by its **name, description, schema and output**.
> You're designing an interface for a very literal user with a limited reading budget.

## 0. Setup

We use **OpenAI** (`gpt-6-luna`) by default. Set `PROVIDER = "anthropic"` to run everything on Claude instead.
Your key comes from the course's `.env` file, as in week 1 (see the README).

Three small modules sit next to this notebook. Read them, they're short:
- `lifedb.py` writes Alex's week into `data/` (dated to **this** week, from the course's `life/` template) and builds
  `life.db` (SQLite) from it, plus a year of generated history.
- `providers.py` is last week's adapters: one interface for OpenAI, Claude and Ollama.
- `agent_core.py` is **last week's loop, as a module** (homework 8.1), with three additions we study below.

In [13]:
%run ../check_setup.py --no-api     # everything should be ✅ (see SETUP.md)


Checking setup for the course in C:\MyWork\Tech-Work\building-agents

✅ Python 3.13
❌ Running in a different environment: C:\MyWork\Tech-Work\building-agents\venv
   → In VS Code: Select Kernel (top right) → Python Environments → .venv. In a terminal: source .venv/bin/activate
✅ Packages installed: openai, pydantic, python-dotenv, ipykernel
✅ OPENAI_API_KEY found in .env (sk-…6pEA)

⚠️  1 problem(s). Fix them with the → hints, run this again, and if it still fails, send this whole output to your instructor before class.


SystemExit: 1

In [ ]:
# Only if the check above says packages are missing:
%pip install -q openai anthropic pydantic python-dotenv


In [1]:
import inspect, json, re, sqlite3, time
from datetime import date as _date
from types import SimpleNamespace
from typing import Annotated, Literal

from pydantic import AfterValidator, BaseModel, ConfigDict, Field, model_validator

import agent_core, lifedb
from agent_core import READ, WRITE, DESTRUCTIVE, FatalToolError, tool, run_agent, execute_tool, first_tool
from providers import load_api_key

PROVIDER = "openai"           # or "anthropic"
MODEL = {"openai": "gpt-6-luna", "anthropic": "claude-sonnet-5-5"}[PROVIDER]   # many runs today: cheap models
load_api_key(PROVIDER)        # reads your key from the course's .env file
DB = lifedb.create_db()       # this week's data + database; run again any time to reset
WEEK = lifedb.week()          # Alex's week follows the real calendar (see life/simweek.py)
TODAY = WEEK.today            # the simulated "now", e.g. "Monday 2026-10-12"

SYSTEM = (
    "You are a personal assistant for Alex, a 44-year-old software developer with a full-time job and two kids, "
    f"who is retraining to become an AI engineer. Today is {TODAY}. "
    "Use the tools to look at Alex's calendar, tasks and notes; never guess. "
    "Before changing anything, make sure you have the right event or task id. Reply briefly."
)

def show(obj):
    print(inspect.getsource(obj))

def call(tools, tool_name, **args):
    # Try a tool by hand, exactly as the loop would run it.
    return execute_tool(SimpleNamespace(id="manual", name=tool_name, input=args), tools)

conn = lifedb.connect(DB, readonly=True)
print("events:", conn.execute("SELECT COUNT(*) FROM events").fetchone()[0],
      "| this week:", conn.execute("SELECT COUNT(*) FROM events WHERE date >= ?", (WEEK.day(0),)).fetchone()[0],
      "| tasks:", conn.execute("SELECT COUNT(*) FROM tasks").fetchone()[0])
conn.close()


events: 543 | this week: 19 | tasks: 9


## 1. Granularity: one tool or many?

Three ways to give the assistant access to the same database:

| Design | Tools | Bet |
|---|---|---|
| **A. One powerful tool** | `run_sql(query)` | The model knows SQL; let it do anything (read-only) |
| **B. Many narrow tools** | `list_events`, `list_tasks`, `free_slots`, `read_note` | Each tool does one job well, with the logic in **our** code |
| **C. Mix** | B + `run_sql` | Narrow tools for common jobs, SQL for the rest |

First, a few helpers and argument types. The `tool()` helper (in `agent_core`) turns a **Pydantic model** into
both the JSON schema the model sees *and* the validation our code runs:

In [2]:
show(agent_core.tool)


def tool(fn: Callable, *, description: str, input_model: type[BaseModel], effect: str = READ,
         name: str | None = None) -> dict:
    """Build a registry entry. The Pydantic model is the single source of truth:
    the model sees its JSON schema, and our code validates every call against it.
    The tool's name (what the model calls) is the function's name, unless you pass `name`."""
    def run(**kwargs: Any) -> str:
        args = input_model.model_validate(kwargs)       # raises ValidationError on bad input
        return str(fn(**dict(args)))
    return {
        "fn": run,
        "effect": effect,
        "schema": {"name": name or fn.__name__, "description": description,
                   "input_schema": input_model.model_json_schema()},
    }



In [3]:
# ---------- argument types: the format rules live in ONE place ----------
def check_date(v: str) -> str:
    try:
        _date.fromisoformat(v)
    except ValueError:
        raise ValueError(f"'{v}' is not a date. Use YYYY-MM-DD, e.g. '{WEEK.day(1)}' (today is {TODAY})") from None
    return v

def check_time(v: str) -> str:
    if not re.fullmatch(r"([01]\d|2[0-3]):[0-5]\d", v):
        raise ValueError(f"'{v}' is not a time. Use 24h HH:MM, e.g. '14:30'")
    return v

Date = Annotated[str, AfterValidator(check_date), Field(description="YYYY-MM-DD")]
Time = Annotated[str, AfterValidator(check_time), Field(description="HH:MM, 24-hour clock")]

class Args(BaseModel):
    model_config = ConfigDict(extra="forbid")   # unknown fields are an error, not silently ignored

# ---------- helpers ----------
MAX_ROWS = 40

def db_rows(sql, params=()):
    if not DB.exists():                               # see Part 4: some errors must stop the run
        raise FatalToolError(f"database not found at {DB}")
    conn = lifedb.connect(DB, readonly=True)          # read-only: nothing the model sends can change data
    try:
        return conn.execute(sql, params).fetchall()
    finally:
        conn.close()

def table(rows, columns, limit=MAX_ROWS, hint=""):
    # Compact TSV, capped. The last line tells the model HOW to get the rest (Part 3).
    if not rows:
        return "(no rows)"
    lines = ["\t".join(columns)] + ["\t".join(str(r[c]) for c in columns) for r in rows[:limit]]
    if len(rows) > limit:
        lines.append(f"... showing {limit} of {len(rows)} rows. {hint}".rstrip())
    return "\n".join(lines)

def minutes(hhmm):
    h, m = map(int, hhmm.split(":"))
    return h * 60 + m

def hhmm(mins):
    return f"{mins // 60:02d}:{mins % 60:02d}"

EVENT_COLS = ["id", "date", "start", "end", "title", "category", "location"]
TASK_COLS = ["id", "title", "course", "due", "est_hours", "priority", "status"]


In [4]:
# ---------- A: one powerful tool ----------
class RunSqlIn(Args):
    query: str = Field(description="One SQLite SELECT statement")

def run_sql(query: str) -> str:
    rows = db_rows(query)
    return table(rows, list(rows[0].keys()) if rows else [], hint="Add a WHERE / LIMIT, or aggregate with COUNT / GROUP BY.")

SQL_DESCRIPTION = ("Read-only SQL over Alex's calendar and tasks. Tables: events(id, date, start, end, title, "
                   "category, location) with date YYYY-MM-DD and times HH:MM; tasks(id, title, course, due, est_hours, "
                   "priority, status) with status 'todo' or 'done'.")

# ---------- B: narrow tools ----------
class ListEventsIn(Args):
    start_date: Date
    end_date: Date

def list_events(start_date: str, end_date: str) -> str:
    rows = db_rows("SELECT * FROM events WHERE date BETWEEN ? AND ? ORDER BY date, start", (start_date, end_date))
    return table(rows, EVENT_COLS, hint="Narrow the date range, or use run_sql with COUNT/GROUP BY.")

class ListTasksIn(Args):
    status: Literal["todo", "done", "all"] = "todo"
    due_before: Date | None = Field(None, description="Only tasks due strictly before this date")

def list_tasks(status: str = "todo", due_before: str | None = None) -> str:
    sql, params = "SELECT * FROM tasks WHERE 1=1", []
    if status != "all":
        sql, params = sql + " AND status = ?", params + [status]
    if due_before:
        sql, params = sql + " AND due < ?", params + [due_before]
    rows = db_rows(sql + " ORDER BY due", tuple(params))
    total = sum(r["est_hours"] for r in rows)
    return table(rows, TASK_COLS) + f"\nTOTAL est_hours: {total:g} ({len(rows)} tasks)"   # what the agent needs next

class FreeSlotsIn(Args):
    date: Date
    day_start: Time = "06:00"
    day_end: Time = "22:30"
    min_minutes: int = Field(30, ge=5, le=600, description="Shortest slot worth reporting")

def free_slots(date: str, day_start: str = "06:00", day_end: str = "22:30", min_minutes: int = 30) -> str:
    # Interval arithmetic belongs in code, not in the model's head (remember Tuesday's overlap?)
    lo, hi = minutes(day_start), minutes(day_end)
    busy = sorted((max(minutes(r["start"]), lo), min(minutes(r["end"]), hi))
                  for r in db_rows("SELECT start, end FROM events WHERE date = ?", (date,)))
    slots, cursor = [], lo
    for start, end in busy:
        if end <= start:
            continue
        if start - cursor >= min_minutes:
            slots.append((cursor, start))
        cursor = max(cursor, end)
    if hi - cursor >= min_minutes:
        slots.append((cursor, hi))
    if not slots:
        return f"No free slot of {min_minutes}+ min on {date} between {day_start} and {day_end}."
    return "\n".join(f"{hhmm(s)}-{hhmm(e)} ({(e - s) / 60:g} h)" for s, e in slots) + \
           f"\nTOTAL free: {sum(e - s for s, e in slots) / 60:g} h"

NOTES = sorted(p.name for p in (lifedb.DATA_DIR / "notes").glob("*.txt"))

def check_note(v: str) -> str:
    if v not in NOTES:
        raise ValueError(f"Unknown note '{v}'. Choose one of: {', '.join(NOTES)}")
    return v

class ReadNoteIn(Args):
    # an enum in the schema: the model sees exactly which notes exist; the validator enforces it
    name: Annotated[str, AfterValidator(check_note), Field(json_schema_extra={"enum": [*NOTES]})]

def read_note(name: str) -> str:
    return (lifedb.DATA_DIR / "notes" / name).read_text(encoding="utf-8")

# ---------- the three designs ----------
SQL_ONLY = {"run_sql": tool(run_sql, description=SQL_DESCRIPTION, input_model=RunSqlIn)}
NARROW = {
    "list_events": tool(list_events, input_model=ListEventsIn, description=
        "List calendar events between two dates (inclusive), ordered by time. Shows each event's id."),
    "list_tasks": tool(list_tasks, input_model=ListTasksIn, description=
        "List tasks (default: open ones), optionally only those due before a date. Includes the total estimated hours."),
    "free_slots": tool(free_slots, input_model=FreeSlotsIn, description=
        "Free time on one day between day_start and day_end (default 06:00-22:30), overlapping events already merged. "
        "Use this instead of computing free time yourself."),
    "read_note": tool(read_note, input_model=ReadNoteIn, description=
        "Read one of Alex's notes: AI course syllabus, personal preferences, job-search pipeline."),
}
MIXED = {**NARROW, **SQL_ONLY}

print(call(NARROW, "free_slots", date=WEEK.day(1)).content)          # Tuesday
print()
print(call(NARROW, "list_tasks", due_before=WEEK.day(4)).content)     # before Friday


06:00-09:00 (3 h)
17:30-19:30 (2 h)
21:00-22:30 (1.5 h)
TOTAL free: 6.5 h

id	title	course	due	est_hours	priority	status
3	Course quiz: embeddings and vector search	AI course	2026-10-20	1.0	medium	todo
4	Tailor resume for the Northwind AI role	Job search	2026-10-20	1.5	high	todo
5	Prepare STAR stories for interviews	Job search	2026-10-21	1.0	medium	todo
7	Sign Lily's field-trip form	Family	2026-10-21	0.25	high	todo
1	Finish the RAG pipeline for the capstone	AI course	2026-10-22	4.0	high	todo
2	Record the capstone demo video	AI course	2026-10-22	1.5	high	todo
TOTAL est_hours: 9.25 (6 tasks)


Now the same three tasks with each design. The checks compare against answers we know from week 1.

In [9]:
TASKS = [
    ("What do I have on Wednesday?",
     lambda a: all(w in a.lower() for w in ("dentist", "open house", "mock interview"))),
    ("Which open tasks are due before Friday, and how many hours of work are they in total?",
     lambda a: "9.25" in a),
    ("Find me a 2-hour block before Thursday night to finish the RAG pipeline for my capstone, "
     "in my best focus hours if possible.",
     None),   # judge this one yourself: best focus is 05:30-06:30 and 20:30-22:30 (preferences.txt).
              # Is there a free 2 h block inside them? (no!) What did it propose instead?
]

def compare(designs, tasks=TASKS, **kw):
    rows = []
    for name, tools in designs.items():
        for i, (task, check) in enumerate(tasks, 1):
            r = run_agent(task, tools, system=SYSTEM, provider=PROVIDER, model=MODEL, verbose=False, **kw)
            ok = "?" if check is None else ("✓" if check(r.answer) else "✗")
            rows.append((name, i, r.turns, len(r.tool_calls), r.input_tokens, ok, r.answer))
    print(f"{'design':8} task turns calls  in_tokens ok")
    for name, i, turns, calls, tin, ok, _ in rows:
        print(f"{name:8} {i:4} {turns:5} {calls:5} {tin:10,} {ok}")
    return rows

rows = compare({"A sql": SQL_ONLY, "B narrow": NARROW, "C mixed": MIXED})


design   task turns calls  in_tokens ok
A sql       1     2     1        574 ✓
A sql       2     2     1        678 ✓
A sql       3     2     2      1,227 ?
B narrow    1     2     1      1,084 ✓
B narrow    2     2     1      1,187 ✓
B narrow    3     3     7      4,564 ?
C mixed     1     2     1      1,266 ✓
C mixed     2     2     1      1,369 ✓
C mixed     3     3    10      4,880 ?


In [10]:
for name, i, *_, answer in rows:
    if i == 3:
        print(f"── {name}\n{answer}\n")


── A sql
Tuesday, Oct 20, **7–9 a.m.** is an open 2-hour block before work, with no calendar conflicts. I can’t verify your preferred focus hours from the available data, but it’s a good uninterrupted morning option. The RAG pipeline is due Thursday and estimated at 4 hours.

── B narrow
Best fit: **tonight, Monday 20:45–22:45**. It’s free, overlaps your preferred evening focus window for 1h 45m, and ends before your 23:00 bedtime. That gives you a solid first block on the RAG pipeline, estimated at 4 hours total.

── C mixed
There’s no 2-hour opening entirely within your stated focus hours before Thursday night. Best fit: **Wednesday, Oct 21, 5:30–7:30 AM**. The first hour is in your early focus window; you’re free until your 8:00 dentist appointment.



**Discuss:**
- Task 3 needs the free time *and* the preferences note. Which design found the "best focus hours"
  (05:30–06:30 and 20:30–22:30)? Did any design notice that 17:30–19:30 is free in the calendar but is really
  family dinner time? **A calendar never shows everything.**
  With SQL only, did the model compute free slots correctly around Tuesday's overlap?
- Count tokens: narrow tools return small, pre-digested answers. SQL returns whatever the query returns.
- When does `run_sql` earn its place? (Try: *"How many gym sessions did I have in March?"* with design B.)
- **Real-world parallel:** Claude Code has `Read`, `Edit`, `Grep`, `Glob` *and* `Bash`. Why not just `Bash`?

## 2. Schemas are guardrails

Our Pydantic models do two jobs. The model sees their **JSON schema**:

In [11]:
print(json.dumps(NARROW["free_slots"]["schema"]["input_schema"], indent=1))


{
 "additionalProperties": false,
 "properties": {
  "date": {
   "description": "YYYY-MM-DD",
   "title": "Date",
   "type": "string"
  },
  "day_start": {
   "default": "06:00",
   "description": "HH:MM, 24-hour clock",
   "title": "Day Start",
   "type": "string"
  },
  "day_end": {
   "default": "22:30",
   "description": "HH:MM, 24-hour clock",
   "title": "Day End",
   "type": "string"
  },
  "min_minutes": {
   "default": 30,
   "description": "Shortest slot worth reporting",
   "maximum": 600,
   "minimum": 5,
   "title": "Min Minutes",
   "type": "integer"
  }
 },
 "required": [
  "date"
 ],
 "title": "FreeSlotsIn",
 "type": "object"
}


…and **every call is validated** before our function runs. Try some bad calls by hand. Read the messages as if you were the model:

In [12]:
for name, args in [
    ("free_slots", {"date": "next Tuesday"}),                  # wrong format
    ("free_slots", {"date": WEEK.day(1), "min_minutes": 0}),   # out of range
    ("list_tasks", {"status": "open"}),                        # not in the enum
    ("list_tasks", {"status": "todo", "sort": "due"}),         # unknown field
    ("read_note", {"name": "syllabus"}),                       # enum lists the real names
]:
    print(f"{name}({args})\n   → {call(NARROW, name, **args).content}\n")


free_slots({'date': 'next Tuesday'})
   → Invalid arguments. date: Value error, 'next Tuesday' is not a date. Use YYYY-MM-DD, e.g. '2026-10-20' (today is Monday 2026-10-19)

free_slots({'date': '2026-10-20', 'min_minutes': 0})
   → Invalid arguments. min_minutes: Input should be greater than or equal to 5

list_tasks({'status': 'open'})
   → Invalid arguments. status: Input should be 'todo', 'done' or 'all'

list_tasks({'status': 'todo', 'sort': 'due'})
   → Invalid arguments. sort: Extra inputs are not permitted

read_note({'name': 'syllabus'})
   → Invalid arguments. name: Value error, Unknown note 'syllabus'. Choose one of: course_syllabus.txt, job_search.txt, preferences.txt



**Good validation errors say what to do next:** the format, an example, today's date, the allowed values.

**Why validate in our code when the APIs have a `strict` mode?** Strict mode makes the model's arguments match the
schema's *shape*. But it also demands that **every** field is required, so optional fields with defaults (like
`day_start`) don't fit. That's why `providers.py` sends OpenAI tools with `strict: false`. We validate ourselves because:
1. It works with **every provider**. Remember the week 1 router: OpenAI and local models get the same checks.
2. Schemas can't express everything: "end after start", "date exists", "no overlap" (Part 5).
3. It's our safety boundary. Never trust input, even from your own model.

**Experiment:** the model resolves "tomorrow" because the system prompt says what today is.

In [14]:
r = run_agent("Am I free tomorrow afternoon?", NARROW, system=SYSTEM, provider=PROVIDER, model=MODEL)
print("\n" + r.answer)



── turn 1 · stop=tool_use · in=465 out=88
  🔧 list_events({"start_date": "2026-10-20", "end_date": "2026-10-20"})
  🔧 free_slots({"date": "2026-10-20", "day_start": "12:00", "day_end": "18:00", "min_minutes": 30})
  ✅ id	date	start	end	title	category	location | 5	2026-10-20	09:00	17:30	Work	work	Home office | 6	2026-10-20	19:30	21:00	AI course: live session (evaluating RAG)	course	Zoom | 7	2026-10-20	20:30	21:00	Re...
  ✅ 17:30-18:00 (0.5 h) | TOTAL free: 0.5 h

── turn 2 · stop=end_turn · in=686 out=99
  💬 You’re booked for work until 5:30 p.m. Tomorrow; you’re free from 5:30 to 6 p.m. in the afternoon window I checked.

You’re booked for work until 5:30 p.m. Tomorrow; you’re free from 5:30 to 6 p.m. in the afternoon window I checked.


In [15]:
# Now hide the date. What does the model pass as `date`? Does validation catch it, or does a *valid but wrong* date slip through?
r = run_agent("Am I free tomorrow afternoon?", NARROW, system=SYSTEM.replace(f" Today is {TODAY}.", ""), provider=PROVIDER, model=MODEL)
print("\n" + r.answer)



── turn 1 · stop=end_turn · in=454 out=133
  💬 What date do you mean by “tomorrow”? I don’t have today’s date available to check your calendar.

What date do you mean by “tomorrow”? I don’t have today’s date available to check your calendar.


**Lesson:** validation catches *malformed* input, not *wrong* input. A valid date can still be the wrong day.
Context (here: today's date) is part of tool design.

## 3. What should a tool return?

Every tool result goes into the context **and is re-sent on every following turn** (week 1, Part 6).
The database holds a year of history. Compare an "honest" tool that returns everything with ours:

In [16]:
def list_events_raw(start_date: str, end_date: str) -> str:
    rows = db_rows("SELECT * FROM events WHERE date BETWEEN ? AND ? ORDER BY date, start", (start_date, end_date))
    return json.dumps([dict(r) for r in rows], ensure_ascii=False)        # everything, as JSON

RAW = {"list_events_raw": tool(list_events_raw, input_model=ListEventsIn,
                               description="List calendar events between two dates (inclusive).")}

year = dict(start_date=WEEK.day(-406), end_date=WEEK.day(6))          # all of the history
raw_out = call(RAW, "list_events_raw", **year).content
capped_out = call(NARROW, "list_events", **year).content
print(f"raw JSON : {len(raw_out):7,} chars ≈ {len(raw_out) // 4:6,} tokens")
print(f"capped   : {len(capped_out):7,} chars ≈ {len(capped_out) // 4:6,} tokens")
print("\nlast line of the capped output:\n ", capped_out.splitlines()[-1])


raw JSON :  75,580 chars ≈ 18,895 tokens
capped   :   2,320 chars ≈    580 tokens

last line of the capped output:
  ... showing 40 of 543 rows. Narrow the date range, or use run_sql with COUNT/GROUP BY.


In [18]:
TASK_GYM = "How many gym sessions did I do per month this year?"
truth = db_rows("SELECT substr(date, 1, 7) AS month, COUNT(*) AS n FROM events "
              "WHERE title = 'Gym' AND date >= ? AND date < ? GROUP BY month", (f"{WEEK.monday.year}-01-01", WEEK.day(0)))
print("ground truth:", {r["month"]: r["n"] for r in truth})

for name, tools in {"raw": RAW, "mixed": MIXED}.items():
    r = run_agent(TASK_GYM, tools, system=SYSTEM, provider=PROVIDER, model=MODEL, verbose=False)
    print(f"\n── {name}: {r.turns} turns, {r.input_tokens:,} input tokens, calls={r.tool_calls}\n{r.answer}")


ground truth: {'2026-01': 8, '2026-02': 5, '2026-03': 8, '2026-04': 8, '2026-05': 8, '2026-06': 5, '2026-07': 3, '2026-08': 6, '2026-09': 6, '2026-10': 4}

── raw: 2 turns, 20,468 input tokens, calls=['list_events_raw']
Your calendar shows **62 gym sessions** so far this year (through Oct 19):

| Month | Sessions |
|---|---:|
| Jan | 8 |
| Feb | 5 |
| Mar | 8 |
| Apr | 8 |
| May | 8 |
| Jun | 5 |
| Jul | 3 |
| Aug | 6 |
| Sep | 6 |
| Oct (through 19th) | 5 |

These are calendar entries, so they don’t confirm whether each session was completed.

── mixed: 3 turns, 2,275 input tokens, calls=['run_sql', 'run_sql']
Based on calendar entries titled “gym,” through today (Oct 19): Jan 8, Feb 5, Mar 8, Apr 8, May 8, Jun 5, Jul 3, Aug 6, Sep 6, and Oct 5.


**Output design rules of thumb**

| Instead of… | Return… |
|---|---|
| every column as JSON | the columns the agent needs, compact (TSV) |
| 500 rows | 40 rows + *"showing 40 of 500, narrow the range or use COUNT/GROUP BY"* |
| raw rows the agent must add up | the total too (`TOTAL est_hours: 9.25`) |
| `"Error"` | `"No event with id 99. Use list_events to find the right id."` |
| an internal stack trace | what went wrong + how to fix it, in one line |

## 4. Three kinds of errors, three owners

| Kind | Example | Who handles it | How |
|---|---|---|---|
| **Model-fixable** | bad date, unknown id, SQL typo | the **model** | return the error *with a hint* |
| **Transient** | "database is locked", network blip | **our code** | retry with backoff; the model never sees it |
| **Fatal** | database missing, credentials revoked | **nobody in the loop** | stop the run, tell the human |

Plus a fourth problem: a tool that **hangs**.

### 4a. Model-fixable: give the model what it needs to fix it

`run_sql` currently passes SQLite's error straight through. What does the model need to write a correct query?
The table and column names. Where can it get them? Three setups:

| Setup | Schema in the description? | Schema in the error message? |
|---|---|---|
| **no schema** | ✗ | ✗ (SQLite's bare error) |
| **schema in errors** | ✗ | ✓ |
| **schema in description** | ✓ | ✓ |

First, the two error messages side by side. Read them as if you were the model:

In [22]:
def run_sql_with_hint(query: str) -> str:
    try:
        return run_sql(query)
    except sqlite3.Error as exc:
        raise ValueError(f"SQL error: {exc}. Tables: events({', '.join(EVENT_COLS)}); "
                         f"tasks({', '.join(TASK_COLS)})") from None

BLIND = "Run a read-only SQL query on Alex's database."          # no schema in the description!
# name="run_sql": the model always sees a tool called run_sql, whichever function runs behind it
SQL_SETUPS = {
    "no schema":             {"run_sql": tool(run_sql, name="run_sql", description=BLIND, input_model=RunSqlIn)},
    "schema in errors":      {"run_sql": tool(run_sql_with_hint, name="run_sql", description=BLIND, input_model=RunSqlIn)},
    "schema in description": {"run_sql": tool(run_sql_with_hint, name="run_sql", description=SQL_DESCRIPTION,
                                              input_model=RunSqlIn)},
}

print(call(SQL_SETUPS["no schema"], "run_sql", query="SELECT starts FROM event").content)
print(call(SQL_SETUPS["schema in errors"], "run_sql", query="SELECT starts FROM event").content)


OperationalError: no such table: event
ValueError: SQL error: no such table: event. Tables: events(id, date, start, end, title, category, location); tasks(id, title, course, due, est_hours, priority, status)


In [21]:
TASK_COURSE = "How many hours did I spend in live AI course sessions in the last 4 weeks?"
truth = db_rows("SELECT SUM((CAST(substr(end,1,2) AS REAL) + substr(end,4,2)/60.0) - "
              "(CAST(substr(start,1,2) AS REAL) + substr(start,4,2)/60.0)) AS h FROM events "
              "WHERE title LIKE 'AI course%' AND date >= ? AND date < ?", (WEEK.day(-28), WEEK.day(0)))[0]["h"]
print("ground truth:", truth, "h")

for name, tools in SQL_SETUPS.items():
    r = run_agent(TASK_COURSE, tools, system=SYSTEM, provider=PROVIDER, model=MODEL, verbose=False)
    print(f"\n── {name}: {r.turns} turns ({len(r.tool_calls)} queries), {r.input_tokens:,} input tokens\n{r.answer}")


ground truth: 12.0 h

── no schema: 5 turns (4 queries), 3,688 input tokens
You spent **12 hours** in live AI course sessions over the last four weeks—8 sessions of 1.5 hours each.

── schema in errors: 4 turns (3 queries), 1,433 input tokens
You spent **12 hours** in live AI course sessions over the last four weeks—eight 90-minute sessions.

── schema in description: 4 turns (3 queries), 3,503 input tokens
You spent **12 hours** in live AI course sessions over the last four weeks (8 sessions).


**Discuss:**
- Count the queries. Without a schema, a resourceful model **discovers** it: it asks SQLite itself
  (`SELECT ... FROM sqlite_master`) or guesses and reads the error. That works, but costs extra turns.
  Run one setup with `verbose=True` to see its queries.
- A good error message rescues a bad guess. A good description avoids the guess altogether.
- Results vary from run to run: run the cell twice. Which setup is *consistently* cheapest?

### 4b. Transient: retry in code, not in the model

A flaky tool fails 60% of the time with `database is locked`. If the model sees every failure, it wastes turns
(and may give up). Our code should retry **idempotent** calls before the model ever hears about it.

In [ ]:
import random

class TransientError(Exception):
    pass

def flaky(fn, fail_rate=0.6, seed=1):
    rng = random.Random(seed)
    def wrapper(**kwargs):
        if rng.random() < fail_rate:
            raise TransientError("database is locked")
        return fn(**kwargs)
    return wrapper

def with_retries(fn, attempts=4, base_delay=0.1):
    def wrapper(**kwargs):
        for attempt in range(1, attempts + 1):
            try:
                return fn(**kwargs)
            except TransientError as exc:
                if attempt == attempts:   # out of retries: NOW it's the model's problem, phrased usefully
                    raise RuntimeError(f"temporarily unavailable after {attempts} attempts ({exc}); "
                                       "tell the user and try again later") from exc
                time.sleep(base_delay * 2 ** (attempt - 1))   # exponential backoff
    return wrapper

def variant(tools, wrap):
    return {name: {**entry, "fn": wrap(entry["fn"])} for name, entry in tools.items()}

FLAKY = variant(NARROW, flaky)
RETRIED = variant(NARROW, lambda fn: with_retries(flaky(fn)))

task = "What do I have on Monday and Tuesday, and which tasks are due before Wednesday?"
for name, tools in {"flaky": FLAKY, "flaky + retries": RETRIED}.items():
    print(f"\n════ {name}")
    r = run_agent(task, tools, system=SYSTEM, provider=PROVIDER, model=MODEL)


Only retry **idempotent** operations: reads, or writes that are safe to repeat. Retrying `create_event` after a timeout could
create the event twice. Our `complete_task` (Part 5) is written so that repeating it is harmless.

### 4c. Fatal: stop the run

Some failures can't be fixed by trying again or by rephrasing. If the database is gone, every tool will fail,
and letting the model loop around that just burns tokens. `db_rows()` raises `FatalToolError`, and the loop stops:

In [ ]:
show(agent_core.execute_tool)


In [ ]:
backup = DB.rename(DB.with_suffix(".bak"))          # simulate: the database disappeared
try:
    r = run_agent("What do I have on Wednesday?", NARROW, system=SYSTEM, provider=PROVIDER, model=MODEL)
    print(f"\nstop={r.stop!r} after {r.turns} turn(s): {r.answer}")
finally:
    backup.rename(DB)


### 4d. A tool that hangs

Without a timeout, one stuck tool freezes the whole agent. Python can't kill a thread, so the work keeps running
in the background; but **the agent** moves on. (Production agents run risky tools in subprocesses they *can* kill.)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeout

_timeout_pool = ThreadPoolExecutor(max_workers=4)

def with_timeout(fn, seconds=2.0):
    def wrapper(**kwargs):
        future = _timeout_pool.submit(fn, **kwargs)
        try:
            return future.result(timeout=seconds)
        except FutureTimeout:
            raise TimeoutError(f"tool timed out after {seconds:g}s; try a narrower request") from None
    return wrapper

def sleepy(fn, seconds=5.0):
    def wrapper(**kwargs):
        time.sleep(seconds)
        return fn(**kwargs)
    return wrapper

t0 = time.perf_counter()
print(call(variant(NARROW, lambda fn: with_timeout(sleepy(fn), 1.0)), "list_tasks").content)
print(f"returned after {time.perf_counter() - t0:.1f}s")


## 5. Side effects & permissions

Now the assistant gets tools that **change** the calendar. Each tool declares an **effect**:

| Effect | Tools | Default policy |
|---|---|---|
| `read` | list_events, list_tasks, free_slots, read_note, run_sql | **allow** |
| `write` | create_event, move_event, complete_task | **ask** |
| `destructive` | cancel_event | **ask** (or **deny**) |

The write tools validate *meaning* as well as format: no overlaps, ids must exist, end after start.

In [ ]:
RW = lifedb.connect(DB)     # ONE shared write connection (autocommit unless we open a transaction)

Category = Literal["work", "course", "job_search", "networking", "family", "health", "personal", "social"]

def describe(e):
    return f"#{e['id']} '{e['title']}' {e['date']} {e['start']}-{e['end']}"

def conflicts(day, start, end, ignore_id=None):
    rows = RW.execute("SELECT * FROM events WHERE date = ? AND start < ? AND end > ? ORDER BY start",
                      (day, end, start)).fetchall()
    return [r for r in rows if r["id"] != ignore_id]

def get_event(event_id):
    row = RW.execute("SELECT * FROM events WHERE id = ?", (event_id,)).fetchone()
    if row is None:
        raise ValueError(f"No event with id {event_id}. Use list_events to find the right id.")
    return row

class Slot(Args):
    date: Date
    start: Time
    end: Time

    @model_validator(mode="after")
    def end_after_start(self):
        if self.end <= self.start:
            raise ValueError(f"end ({self.end}) must be after start ({self.start})")
        return self

class CreateEventIn(Slot):
    title: str = Field(min_length=1, max_length=80)
    category: Category
    location: str = ""

def create_event(date, start, end, title, category, location=""):
    if clash := conflicts(date, start, end):
        raise ValueError("Conflicts with " + ", ".join(map(describe, clash)) +
                         ". Pick another time (see free_slots) or ask the user what to move.")
    cur = RW.execute("INSERT INTO events (date, start, end, title, category, location) VALUES (?, ?, ?, ?, ?, ?)",
                     (date, start, end, title, category, location))
    return f"Created #{cur.lastrowid} '{title}' {date} {start}-{end}"

class MoveEventIn(Slot):
    event_id: int

def move_event(event_id, date, start, end):
    old = get_event(event_id)
    if clash := conflicts(date, start, end, ignore_id=event_id):
        raise ValueError("Conflicts with " + ", ".join(map(describe, clash)) + ". Pick another time.")
    RW.execute("UPDATE events SET date = ?, start = ?, end = ? WHERE id = ?", (date, start, end, event_id))
    return f"Moved {describe(old)} -> {date} {start}-{end}"

class EventIdIn(Args):
    event_id: int

def cancel_event(event_id):
    old = get_event(event_id)
    RW.execute("DELETE FROM events WHERE id = ?", (event_id,))
    return f"Cancelled {describe(old)}"

class TaskIdIn(Args):
    task_id: int

def complete_task(task_id):
    row = RW.execute("SELECT * FROM tasks WHERE id = ?", (task_id,)).fetchone()
    if row is None:
        raise ValueError(f"No task with id {task_id}. Use list_tasks to find the right id.")
    if row["status"] == "done":
        return f"Task #{task_id} '{row['title']}' was already done."      # idempotent: safe to repeat
    RW.execute("UPDATE tasks SET status = 'done' WHERE id = ?", (task_id,))
    return f"Marked #{task_id} '{row['title']}' as done"

WRITES = {
    "create_event": tool(create_event, input_model=CreateEventIn, effect=WRITE,
                         description="Add an event to the calendar. Fails if it overlaps an existing event."),
    "move_event": tool(move_event, input_model=MoveEventIn, effect=WRITE,
                       description="Move an existing event (by id) to a new date and time. Fails on overlap."),
    "complete_task": tool(complete_task, input_model=TaskIdIn, effect=WRITE,
                          description="Mark a task (by id) as done."),
    "cancel_event": tool(cancel_event, input_model=EventIdIn, effect=DESTRUCTIVE,
                         description="Delete an event (by id) from the calendar. Cannot be undone."),
}
ALL = {**MIXED, **WRITES}

print(call(ALL, "create_event", date=WEEK.day(1), start="20:00", end="20:30", title="Quick call with Priya",
           category="networking").content)


### The policy: allow / ask / deny, with a dry-run

For **ask**, we don't just show the arguments. We **run the change inside a transaction**, show the user the
actual diff, then **commit** or **roll back**. The user approves *what will happen*, not what the model *says* will happen.

We don't touch the loop at all: `apply_policy` wraps each tool's function.

In [ ]:
POLICY = {READ: "allow", WRITE: "ask", DESTRUCTIVE: "ask"}

def ask_human(preview: str) -> bool:
    return input(f"\n{preview}\nAllow? [y/N] ").strip().lower() in ("y", "yes")

def guard(name, fn, decision, approver):
    if decision == "allow":
        return fn
    if decision == "deny":
        def denied(**kwargs):
            raise PermissionError(f"'{name}' is turned off in the user's settings. "
                                  "Tell the user what you wanted to do so they can do it themselves.")
        return denied

    def ask(**kwargs):
        before = lifedb.snapshot(RW)
        RW.execute("BEGIN")                      # dry-run starts
        committed = False
        try:
            output = fn(**kwargs)                # invalid call? it raises here: the user is never bothered
            changes = lifedb.diff(before, lifedb.snapshot(RW))
            args = ", ".join(f"{k}={v!r}" for k, v in kwargs.items())
            if approver(f"🔐 {name}({args})\n{changes}"):
                RW.execute("COMMIT")
                committed = True
                return output
        finally:
            if not committed:                    # declined, invalid, or the prompt itself failed
                RW.execute("ROLLBACK")
        raise PermissionError("The user declined this change. Ask them what they would prefer.")
    return ask

def apply_policy(tools, policy=POLICY, overrides=None, approver=ask_human):
    overrides = overrides or {}
    return {name: {**e, "fn": guard(name, e["fn"], overrides.get(name, policy[e["effect"]]), approver)}
            for name, e in tools.items()}


In [ ]:
# By hand first: answer the prompt with y or n, then check the database.
GUARDED = apply_policy(ALL)
print(call(GUARDED, "move_event", event_id=8, date=WEEK.day(4), start="08:00", end="08:45").content)   # to Friday
print("dentist now:", dict(db_rows("SELECT date, start, end FROM events WHERE id = 8")[0]))


In [ ]:
RW.close(); lifedb.create_db(); RW = lifedb.connect(DB)     # reset, so the agent starts from the original week
GUARDED = apply_policy(ALL)
r = run_agent("Move my dentist appointment on Wednesday to Friday at 8:00, and mark Lily's field-trip form as done.",
              GUARDED, system=SYSTEM, provider=PROVIDER, model=MODEL)
print("\n" + r.answer)


In [ ]:
# The user turned deleting off. What does the agent do with "no"?
STRICT = apply_policy(ALL, overrides={"cancel_event": "deny"})
r = run_agent("Cancel dinner with friends on Saturday.", STRICT, system=SYSTEM, provider=PROVIDER, model=MODEL)
print("\n" + r.answer)


**Discuss:**
1. A dry-run works because a database has transactions. How would you preview **sending an email** or **booking a flight**?
   (Hint: describe the action, and the confirmation *is* the commit. Claude Code shows a diff before every edit.)
2. Answer "n" to a prompt. What does the model do next? Is "The user declined this change. Ask them what they would prefer." a good message?
3. Where do permissions live in real agents? Claude Code has allow/ask/deny rules per tool. pi and OpenClaw have
   `beforeToolCall`/`afterToolCall` hooks, and Hermes makes approval prompts wait their turn so they never interleave.
4. Why does `guard` roll back an **invalid** call *before* asking the user? And why is the rollback in a `finally`?
   (Hint: what if the approval prompt itself crashes, or the user hits Ctrl+C?)

## 6. Parallel tool calls

One model reply can contain several tool calls. If they're independent **reads**, there's no reason to run them one by one:

In [ ]:
show(agent_core.run_tools)


In [ ]:
SLOW = variant(NARROW, lambda fn: sleepy(fn, 0.8))       # pretend every read takes 0.8 s (a real API would)
task = "What are my free slots on Monday, Tuesday, Wednesday and Thursday? Look up all four days at once."

for parallel in (False, True):
    r = run_agent(task, SLOW, system=SYSTEM, provider=PROVIDER, model=MODEL, parallel=parallel, verbose=False)
    print(f"parallel={parallel!s:5}: {r.seconds:5.1f}s total, {r.turns} turns, calls={r.tool_calls}")


**Discuss:** Why does `run_tools` only parallelise when **every** call in the batch is a read?
What could go wrong running `move_event` and `create_event` at the same time? Or two approval prompts?
(Hermes uses a thread pool of up to 8 workers, and makes approval prompts wait their turn.)

## 7. Does the model pick the right tool? A first eval

Descriptions are prompts, so measure them like prompts. For each request we list the tools that are a sensible
**first** move, then ask the model once (nothing is executed) and score.

In [ ]:
EVAL = [
    ("What's on my calendar on Wednesday?",             {"list_events"}),
    ("Which tasks are still open?",                      {"list_tasks"}),
    ("When am I free on Thursday for an hour?",          {"free_slots"}),
    ("What does my course syllabus say about demo day?", {"read_note"}),
    ("How many gym sessions did I have last month?",     {"run_sql"}),
    ("How do I like to organise my week?",               {"read_note"}),
    ("Add a capstone work block Thursday 17:30-19:30",   {"create_event", "free_slots", "list_events"}),
    ("Move my dentist appointment to Friday 8:00",       {"list_events", "move_event"}),
    ("I finished the course quiz",                       {"list_tasks", "complete_task"}),
    ("Cancel dinner with friends",                       {"list_events", "cancel_event"}),
]

def score(tools, label):
    hits = 0
    for request, ok in EVAL:
        picked = first_tool(request, tools, system=SYSTEM, provider=PROVIDER, model=MODEL)
        hits += picked in ok
        print(f"  {'✓' if picked in ok else '✗'} {picked:14} ← {request}")
    print(f"{label}: {hits}/{len(EVAL)}\n")
    return hits

def with_descriptions(tools, text):
    return {n: {**e, "schema": {**e["schema"], "description": text}} for n, e in tools.items()}

def with_opaque_names(tools):
    renamed = {}
    for i, (n, e) in enumerate(tools.items(), 1):
        renamed[f"tool_{i}"] = {**e, "schema": {**e["schema"], "name": f"tool_{i}"}}
    return renamed, {f"tool_{i}": n for i, n in enumerate(tools, 1)}

good = score(ALL, "good names + good descriptions")
vague = score(with_descriptions(ALL, "Calendar helper."), "good names + vague descriptions")


In [ ]:
# Opaque names, good descriptions. Map the names back before scoring.
OPAQUE, back = with_opaque_names(ALL)
hits = 0
for request, ok in EVAL:
    picked = back.get(first_tool(request, OPAQUE, system=SYSTEM, provider=PROVIDER, model=MODEL), "(no tool)")
    hits += picked in ok
    print(f"  {'✓' if picked in ok else '✗'} {picked:14} ← {request}")
print(f"opaque names + good descriptions: {hits}/{len(EVAL)}")


**Discuss:** Which matters more, the name or the description? Ten cases is a tiny eval. How many would you need
before you trust a 1-point difference? (Week 7 is all about this.)

## 8. Homework

### 8.1 Grow your package (required)

Add to the package you built in week 1:

```
agent/
├── db.py            # build / connect / snapshot / diff   (from lifedb.py)
├── tools.py         # life tools: Pydantic inputs, effect per tool, capped outputs, helpful errors
├── permissions.py   # POLICY, overrides, apply_policy, ask_human; dry-run with commit / rollback
├── reliability.py   # TransientError, with_retries (reads only!), with_timeout
└── loop.py          # run_tools: parallel reads, sequential writes; FatalToolError ends the run
```

Tests (no API key; give each test its **own temporary database**):
- `free_slots` on Tuesday merges the overlap: the evening slot starts at `21:00`, `TOTAL free: 6.5 h`
- a bad date returns an error that mentions `YYYY-MM-DD`; unknown fields are rejected
- `run_sql("DELETE FROM events")` fails and deletes nothing
- an approved `move_event` is committed; a declined one leaves the database unchanged
- an invalid write under "ask" **never calls the approver**
- 4 parallel reads of 0.3 s finish in < 0.8 s, results in order; writes run one at a time
- a missing database stops the loop with `stop == "fatal"`

### 8.2 A new write tool (required)

Add `add_task(title, course, due, est_hours, priority)`:
- validation: `due` not in the past (relative to today), `0 < est_hours <= 40`, `priority` is an enum
- effect `write` → goes through "ask"
- then ask the agent: *"Northwind AI sent me a take-home assignment, due next Wednesday, about 5 hours."*

### 8.3 Stretch
- **Undo:** keep a log of every committed change (the diff), and add an `undo_last_change` tool.
- **Cross-provider eval:** run Part 7's eval through the week 1 router on two providers. Are descriptions portable?
- **MCP:** publish your *read* tools as an MCP server and connect it to Claude Code or Claude Desktop.
  The same tools, used by a different agent. (We'll come back to MCP later in the course.)